In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
plt.rcParams.update({
    "font.size" : 15,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "stix",
})


In [ ]:
folderVec = [
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/fwh_monopole_srt_tau06_smol_sphere/microphones/obs1",
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/fwh_monopole_srt_tau06_smol_sphere_d10/microphones/obs1",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/fwh_monopole_srt_tau06_smol_sphere_d20/microphones/obs1",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/fwh_monopole_srt_tau06_smol_sphere_d30/microphones/obs1",
]

obsLblVec = [
    r'$\theta = \pi$',
    r'$\theta = \pi / 2$',
    r'$\theta = 0.0$',
]

colorVec = [
    # "#910000",
    # "#119100",
    "#001AFF",
    '#FF5733',
    "#B700FF",
]

markerVec = [
    'o',
    '^',
    's',
    "P",
    '*',
]


In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=2):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
        vec = vec[~np.isnan(vec)]
    f.close()
    return vec
def get_probe_coords(filePath):
    with open(filePath, 'r') as f:
        header = f.readline()
        coordsStr = header.split('(')[1].split(')')[0].split(',')
        x = float(coordsStr[0])
        y = float(coordsStr[1])
        z = float(coordsStr[2])
    return x,y,z


In [ ]:
# read observers (y)
obsVecVec = []
dataVecVec = []
obsTimeVecVec = []
for case in range(0, len(folderVec)):
    fileList = sorted(glob.glob(f"{folderVec[case]}/microphone*.txt"))
    coordVec = [[], [], []]
    dataVec = []
    obsTimeVec = []
    for filePath in fileList:
        x, y, z = get_probe_coords(filePath)
        coordVec[0].append(x)
        coordVec[1].append(y)
        coordVec[2].append(z)
        dataVec.append(read_col_probe(filePath, 2))
        obsTimeVec.append(read_col_probe(filePath, 1))
    obsVecVec.append(coordVec)
    dataVecVec.append(dataVec)
    obsTimeVecVec.append(obsTimeVec)



In [ ]:
dxPhy = 1.0
rho0Phy = 1.0
# rho0Phy = 1.0 #! test
csPhy = 340.0
# csPhy = 1.0 #! test
gamma = 1.0
csLB = np.sqrt(gamma / 3.0)

CLength = dxPhy / 1.0
CRho = rho0Phy / 1.0
CVel = csPhy / csLB
CTime = CLength / CVel
CMass = CRho * CLength**3
CPressure = CMass / (CLength * CTime**2)

freqStep = 0.05
M0 = 0.0
# A = 1.0 / 340.0 #! test
ampLB = 0.01
ampPhy = ampLB * CRho
# TPhy = 1.0 #! test
TPhy = (1.0 / freqStep) * CTime
print(TPhy)


In [ ]:

U0Phy = M0 * csPhy
omega = 2 * np.pi / TPhy
print(omega)
print(M0)

In [ ]:

#! reference (physical unit)
beta = np.sqrt(1.0 - M0**2)
betaSq = beta**2
print(betaSq)
pSrc = [0, 0, 0]
timeTotal = 10 * TPhy
dt = 0.001
tVec = np.linspace(0.0, timeTotal, int(round(timeTotal / dt)) + 1, True)
rg = [0, 2]
timeRg = [rg[0] * TPhy, rg[1] * TPhy]

uPrimeVecVec = []
pPrimeVecVec = []
for case in range(0, len(folderVec)):
    uPrimeVec = [[], [], []]
    pPrimeVec = []
    for iObs in range(0, len(obsVecVec[case][0])):
        pObs = [obsVecVec[case][0][iObs], obsVecVec[case][1][iObs], obsVecVec[case][2][iObs]]
        RStar = np.sqrt((pObs[0] - pSrc[0])**2 + betaSq * ((pObs[1] - pSrc[1])**2 + (pObs[2] - pSrc[2])**2))
        R = (RStar - M0 * (pObs[0] - pSrc[0])) / betaSq
        RStarDerivX = (pObs[0] - pSrc[0]) / RStar
        RStarDerivY = betaSq * (pObs[1] - pSrc[1]) / RStar
        RStarDerivZ = betaSq * (pObs[2] - pSrc[2]) / RStar
        RDerivX = (RStarDerivX - M0) / betaSq
        RDerivY = RStarDerivY / betaSq
        RDerivZ = RStarDerivZ / betaSq

        factor = ampPhy / (4 * np.pi * RStar) * np.exp(1j * omega * (tVec - R / csPhy))
        uXTmp = RStarDerivX / RStar + 1j * omega / csPhy * RDerivX
        uYTmp = RStarDerivY / RStar + 1j * omega / csPhy * RDerivY
        uZTmp = RStarDerivZ / RStar + 1j * omega / csPhy * RDerivZ
        uPrime = [-factor * uXTmp, -factor * uYTmp, -factor * uZTmp]
        uPrimeVec[0].append(np.real(uPrime[0]))
        uPrimeVec[1].append(np.real(uPrime[1]))
        uPrimeVec[2].append(np.real(uPrime[2]))
        pPrimeVec.append(np.real(-rho0Phy * (1j * omega * factor + U0Phy * uPrime[0])))
    uPrimeVecVec.append(uPrimeVec)
    pPrimeVecVec.append(pPrimeVec)

startIdx = int(round(timeRg[0] / dt))
endIdx = int(round(timeRg[1] / dt))


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(16, 6), facecolor='w', edgecolor='w')
# for case in range(1, 2):
for case in range(0, len(folderVec)):
    # for iObs in range(0, len(obsVecVec[case][0])):
    for iObs in range(0, 1):
        # timeVec = np.linspace(rg[0], rg[1], len(tVec[startIdx:endIdx]), True)
        timeVec = tVec
        # ax.plot(timeVec+0.192, pPrimeVecVec[case][iObs][startIdx:endIdx], label=obsLblVec[iObs], marker=markerVec[iObs], fillstyle='none')
        ax.plot(timeVec+0.195, pPrimeVecVec[case][iObs], label=obsLblVec[iObs], marker=markerVec[iObs], fillstyle='none')
        ax.plot(obsTimeVecVec[case][iObs], dataVecVec[case][iObs]*0.0005, marker=markerVec[iObs])
# ax.set_xlim(rg[0], rg[1])
ax.set_xlim(0.15, 0.35)
# ax.set_ylim(-2E-04, 2E-04)
# ax.set_ylim(-0.02, 0.02)
ax.set_xlabel(r'$t/T$')
ax.set_ylabel(r'$p\prime$')
ax.legend(loc='upper center', frameon=False)